# VQ-VAE-2 Latents — Probes

Information-leakage, diagnosis, demographic, and L0-leakage probes on content vs style features.

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = (
    "/home/ng24/projects/multiview-crl/results/ADNI_registered/multiview-06-content-lr-002-single-level/vqvae_best1.pt"
)
DATA_DIR = "/data/natalia/ADNI_stripped"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 200  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

# ── Auto-detect content/style split from checkpoint weights ──────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Strip MoCo "online." prefix if present to find the right keys
_prefix = ""
if any(k.startswith("online.") for k in state_dict):
    _prefix = "online."

# channel_logits is now a ParameterDict keyed by level.
# Old checkpoints: "module.channel_logits" (single param)
# New checkpoints: "module.channel_logits.0", "module.channel_logits.1", etc.
_cl_key_old = f"{_prefix}module.channel_logits"

# Find all channel_logits.X keys where X is a digit
_cl_levels = []
for k in state_dict:
    stripped = k[len(_prefix) :] if k.startswith(_prefix) else k
    if stripped.startswith("module.channel_logits."):
        suffix = stripped[len("module.channel_logits.") :]
        if suffix.isdigit():
            _cl_levels.append(int(suffix))
_cl_levels = sorted(_cl_levels)

# ── Detect mask_mode from checkpoint keys ────────────────────────────────────
# fixed mode: has fixed_mask_* buffers, no channel_logits
# learned mode: has channel_logits params
# onthefly mode: neither (logits computed from activations at runtime)
_has_fixed_mask = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.fixed_mask_") for k in state_dict
)

if _has_fixed_mask:
    _detected_mask_mode = "fixed"
elif _cl_levels or _cl_key_old in state_dict:
    _detected_mask_mode = "learned"
else:
    _detected_mask_mode = "onthefly"

# Allow settings.json to override (it's always authoritative if present)
_mask_mode = settings.get("mask_mode", _detected_mask_mode)

if _cl_levels:
    # New-style per-level channel_logits
    _cl_key_0 = f"{_prefix}module.channel_logits.{_cl_levels[0]}"
    _mask_dim = state_dict[_cl_key_0].shape[0]
    content_style_levels = _cl_levels
elif _cl_key_old in state_dict:
    # Old-style single channel_logits
    _mask_dim = state_dict[_cl_key_old].shape[0]
    content_style_levels = [0]
else:
    _mask_dim = None
    # No channel_logits (onthefly or fixed mode) — read content_style_levels from settings
    content_style_levels = settings.get("content_style_levels", [0])

# ── Per-level content_channels detection from codebook input sizes ────────────
hidden_channels = settings["vqvae_hidden_channels"]
_embed_dim = settings["vqvae_embed_dim"]
_nb_levels = settings["vqvae_nb_levels"]

# Detect per-level content_channels from codebook conv_in dimensions.
# Works for any mask_mode (learned, onthefly, or fixed) as long as the codebook was
# sized for the content subset (content_channels < hidden_channels).
_content_ch_per_level = {}
_used_codebook_detection = False
content_ratios = None

for lvl in content_style_levels:
    _cb_key = f"{_prefix}module.codebooks.{lvl}.conv_in.weight"
    if _cb_key in state_dict:
        _cb_in = state_dict[_cb_key].shape[1]
        if lvl == _nb_levels - 1:
            # Top level: codebook input = content_channels (no embed_dim concat)
            _content_ch_per_level[lvl] = _cb_in
        else:
            # Lower levels: codebook input = content_channels + embed_dim
            _content_ch_per_level[lvl] = _cb_in - _embed_dim

# Check if codebook detection is valid: if ALL detected levels give
# content_channels == hidden_channels, the checkpoint predates per-level
# codebook sizing — fall back to settings-based CONTENT_SIZE / STYLE_SIZE.
_all_full_width = all(cc == hidden_channels for cc in _content_ch_per_level.values()) if _content_ch_per_level else True

if not _all_full_width:
    # Codebook was sized for content masking — use detected values
    _used_codebook_detection = True

    _first_lvl = content_style_levels[0]
    _content_channels = _content_ch_per_level.get(_first_lvl, hidden_channels)
    _style_channels = hidden_channels - _content_channels

    CONTENT_SIZE = _content_channels
    STYLE_SIZE = _style_channels

    # Compute per-level content_ratios if levels have different content sizes
    _ratios = [_content_ch_per_level[lvl] / hidden_channels for lvl in content_style_levels]
    if len(set(_ratios)) > 1:
        content_ratios = _ratios

    _mode_desc = {
        "learned": "learned",
        "onthefly": "onthefly (no channel_logits)",
        "fixed": "fixed (first K = content)",
    }
    print(f"Auto-detected from checkpoint (per-level codebook sizing):")
    print(f"  mask_mode             : {_mode_desc.get(_mask_mode, _mask_mode)}")
    print(f"  content_style_levels  : {content_style_levels}")
    for lvl in content_style_levels:
        cc = _content_ch_per_level.get(lvl, "?")
        print(f"  level {lvl} content_ch    : {cc} / {hidden_channels}  ({cc/hidden_channels:.1%})")
    print(f"  -> CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}")
    if content_ratios:
        print(f"  -> content_ratios={content_ratios}")
else:
    # Old checkpoint: codebooks use full hidden_channels.
    # Keep CONTENT_SIZE / STYLE_SIZE from settings cell (cell above).
    print(f"Codebook detection gives content_ch == hidden_channels at all levels.")
    print(f"  content_style_levels  : {content_style_levels}")
    print(f"  → Using CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE} from settings.json")

# ── Detect whether checkpoint uses hidden_channels or embed_dim masking ──────
if _mask_dim is not None and _mask_dim != hidden_channels:
    raise RuntimeError(
        f"Checkpoint has channel_logits of size {_mask_dim} but hidden_channels={hidden_channels}. "
        f"This checkpoint was likely trained with an older code version where the mask was on "
        f"embed_dim={_embed_dim}. You need to checkout the matching code version "
        f"to load this checkpoint."
    )

# ── Detect separate_encoders from checkpoint keys ────────────────────────────
# If the checkpoint contains "module.encoders_v1.*" keys, the model was trained
# with --separate-encoders.
_sep_enc = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.encoders_v1.") for k in state_dict
)
if _sep_enc:
    print("Detected separate_encoders in checkpoint (encoders_v1 keys found).")

_style_inj_mode = settings.get("style_injection_mode", "concat")

# ── Read training flags that affect the encoder forward pass ─────────────────
# These MUST match training exactly, otherwise the encoder produces different
# features at inference time (e.g. style channels zeroed between levels when
# they should be passed through, causing out-of-distribution inputs at higher levels).
_pass_full = settings.get("pass_full_to_next_level", False)
_narrow_enc = settings.get("narrow_encoder_input", False)
_top_recon = settings.get("top_level_recon_only", False)
_quantize_style = settings.get("quantize_style", False)
_style_embed_dim = settings.get("style_embed_dim", None)
_style_nb_entries_raw = settings.get("style_nb_entries", None)
_use_content_proj = settings.get("use_content_projection", False)

# ── Normalize per-level codebook sizes ───────────────────────────────────────
# --vqvae-nb-entries / --style-nb-entries are argparse nargs="+" → always a
# list in settings.json. W&B sweeps occasionally round-trip these as
# stringified lists ("[384]") or nested lists ([[384]]), which then reach
# CodeLayer where torch.randn(embed_dim, nb_entries) raises
#   TypeError: randn(): argument 'size' failed to unpack ... got list
# Flatten + coerce to int / flat list[int] before handing to VQVAE.
import ast as _ast


def _normalize_entries(v):
    if v is None:
        return None
    if isinstance(v, str):
        try:
            v = _ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return int(v)
    if isinstance(v, (list, tuple)):
        flat = []
        for x in v:
            if isinstance(x, (list, tuple)):
                flat.extend(x)
            else:
                flat.append(x)
        flat = [int(x) for x in flat]
        return flat[0] if len(flat) == 1 else flat
    return int(v)


_nb_entries = _normalize_entries(settings["vqvae_nb_entries"])
_style_nb_entries = _normalize_entries(_style_nb_entries_raw)
print(f"vqvae_nb_entries (normalized) : {_nb_entries}")
if _style_nb_entries is not None:
    print(f"style_nb_entries (normalized) : {_style_nb_entries}")

# ── Optional: skip levels in the final reconstruction decoder ────────────────
# Define SKIP_RECON_LEVELS in a cell above to ablate finer codes, e.g.
#     SKIP_RECON_LEVELS = [0, 1]
# zeros out levels 0 and 1 in the final (level-0) decoder concat, leaving only
# the top (coarsest) codes to drive reconstruction. The top level itself cannot
# be skipped. If undefined, falls back to whatever was stored at training time.
try:
    _skip_recon = SKIP_RECON_LEVELS  # noqa: F821 — optional user override
    print(f"⚠ Notebook override: skip_decoder_concat_levels={_skip_recon}")
except NameError:
    _skip_recon = settings.get("skip_decoder_concat_levels", None)
    if _skip_recon:
        print(f"skip_decoder_concat_levels (from settings): {_skip_recon}")

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=hidden_channels,
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=_nb_levels,
    embed_dim=_embed_dim,
    nb_entries=_nb_entries,
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
    content_style_levels=content_style_levels,
    content_ratios=content_ratios,
    separate_encoders=_sep_enc,
    mask_mode=_mask_mode,
    style_injection_mode=_style_inj_mode,
    pass_full_to_next_level=_pass_full,
    narrow_encoder_input=_narrow_enc,
    top_level_recon_only=_top_recon,
    quantize_style=_quantize_style,
    style_embed_dim=_style_embed_dim,
    style_nb_entries=_style_nb_entries,
    use_content_projection=_use_content_proj,
    skip_decoder_concat_levels=_skip_recon,
)

content_channels = vqvae_model.content_channels
print(f"\nhidden_channels         : {hidden_channels}")
print(f"content_channels        : {content_channels}")
if vqvae_model.content_channels_per_level:
    print(f"content_channels_per_lvl: {vqvae_model.content_channels_per_level}")
print(f"content_style_levels    : {content_style_levels}")
print(f"mask_mode               : {_mask_mode}")
print(f"inject_style_to_decoder : {inject_style}")
print(f"separate_encoders       : {_sep_enc}")
print(f"style_injection_mode    : {_style_inj_mode}")
print(f"pass_full_to_next_level : {_pass_full}")
print(f"narrow_encoder_input    : {_narrow_enc}")
print(f"top_level_recon_only    : {_top_recon}")
print(f"quantize_style          : {_quantize_style}")
print(f"use_content_projection  : {_use_content_proj}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# ── Key remapping ─────────────────────────────────────────────────────────────
new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    # Migrate old single channel_logits → ParameterDict key "0"
    if k.endswith("module.channel_logits") and not any(c.isdigit() for c in k.split(".")[-1]):
        k = k + ".0"
    if k.startswith("online."):
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(
        ("momentum_encoders.", "momentum_codebook_projs.", "momentum_content_proj.", "momentum_level0_proj.", "queue_")
    ):
        pass  # MoCo-only state — discard
    elif k.startswith("momentum_encoders_v1."):
        pass  # MoCo momentum copy of view-1 encoder — discard
    else:
        new_state_dict[k] = v

# ── Drop keys with shape mismatches (e.g. projection head from older config) ─
_model_sd = vqvae_model.state_dict()
_shape_skipped = []
for k in list(new_state_dict):
    if k in _model_sd and new_state_dict[k].shape != _model_sd[k].shape:
        _shape_skipped.append(f"{k}: ckpt {new_state_dict[k].shape} vs model {_model_sd[k].shape}")
        del new_state_dict[k]
if _shape_skipped:
    print(f"\u26a0 Skipped {len(_shape_skipped)} keys with shape mismatch:")
    for s in _shape_skipped:
        print(f"    {s}")

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"\u26a0 Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"\u26a0 Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("\u2713 Checkpoint loaded cleanly")
elif not missing_real:
    print("\u2713 Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"\u2713 Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

In [ ]:
# -- Diagnostic: isolate where the constant-output problem lives --
import torch

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# 1. Check encoder weight statistics (are weights non-trivial?)
print("=== Encoder weight check ===")
for name, p in list(vqvae_module.encoders[0].named_parameters())[:6]:
    print(
        f"  {name}: shape={tuple(p.shape)}  "
        f"mean={p.data.mean():.6f}  std={p.data.std():.6f}  "
        f"absmax={p.data.abs().max():.6f}"
    )

# 2. Check ReZero alpha values (if 0, residual blocks are identity)
print("\n=== ReZero alpha values ===")
for name, p in vqvae_module.encoders[0].named_parameters():
    if "alpha" in name:
        print(f"  {name}: {p.data.item():.6f}")

# 3. Test raw encoder (bypass VQVAE forward)
print("\n=== Raw encoder test ===")
with torch.no_grad():
    d1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    d2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)

    # Run through encoder level 0 directly
    out1 = vqvae_module.encoders[0](d1)
    out2 = vqvae_module.encoders[0](d2)

    print(f"  Encoder output shape: {out1.shape}")
    print(f"  out1 range: [{out1.min():.4f}, {out1.max():.4f}]  mean={out1.mean():.6f}")
    print(f"  out2 range: [{out2.min():.4f}, {out2.max():.4f}]  mean={out2.mean():.6f}")

    # Check spatial variation (different voxels should have different values)
    print(f"  out1 spatial std (per-channel): {out1[0].std(dim=[1, 2, 3]).mean():.6f}")

    # Pool and compare
    p1 = out1.mean(dim=[2, 3, 4])
    p2 = out2.mean(dim=[2, 3, 4])
    cos = torch.nn.functional.cosine_similarity(p1, p2).item()
    print(f"  Pooled cosine similarity: {cos:.6f}")
    if cos > 0.999:
        print("  WARNING: Raw encoder also produces constant output!")
    else:
        print("  OK: Raw encoder works - bug is in VQVAE.forward()")

# 4. Test layer by layer to find where signal dies
print("\n=== Layer-by-layer signal propagation ===")
with torch.no_grad():
    x1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    x2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    for layer_idx, layer in enumerate(vqvae_module.encoders[0].layers):
        x1 = layer(x1)
        x2 = layer(x2)
        cos = torch.nn.functional.cosine_similarity(x1.flatten(1), x2.flatten(1)).item()
        print(
            f"  After layer {layer_idx} ({type(layer).__name__}): "
            f"shape={tuple(x1.shape)}  cos={cos:.6f}  "
            f"range=[{x1.min():.3f},{x1.max():.3f}]"
        )

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
# Match the training val pipeline EXACTLY. Previously the notebook always
# applied CreateBrainMaskd (intensity > 0 threshold). Training with masks on
# disk skips that and loads the proper brain-mask .nii.gz directly via
# LoadImaged — a different mask → different NormalizeIntensityd stats →
# different features → different modality probe result. Fix: delegate to
# utils.utils.transforms and use masks_from_disk when the CSV items carry
# mask paths.
from utils.utils import transforms as get_transforms

# Read spacing and spatial_size from settings.json (saved at training time)
spacing = settings.get("image_spacing", 2.0)
crop_margin = settings.get("crop_margin", 0)

_saved_spatial = settings.get("spatial_size", None)
if _saved_spatial is not None:
    spatial_size = tuple(_saved_spatial)
elif spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

# Mirror data/datasets.py:1283 — masks_from_disk is True iff every loaded
# item carries both mask paths.
masks_from_disk = all(("mask_image" in it) and ("mask_z_image" in it) for it in items)

_, _val_transforms_raw = get_transforms(
    spacing=spacing,
    crop_margin=crop_margin,
    spatial_size=spatial_size,
    masks_from_disk=masks_from_disk,
    asymmetric_aug=False,
)

# The raw val transform expects:
#   - ``mask_t1``/``mask_t2`` paths when ``masks_from_disk=True`` (loaded by LoadImaged)
#   - a ``label`` key at the end (required by the final ToTensord in utils.utils.transforms)
# Existing notebook cells pass only ``image_t1``/``image_t2``. Wrap the
# transform to auto-inject mask paths and a placeholder label, keyed on the
# T1 image path.
_img_to_extras = {
    it["image"]: {
        "mask_t1": it.get("mask_image"),
        "mask_t2": it.get("mask_z_image"),
        "label": it.get("label", 0),
    }
    for it in items
}


def val_transforms(data_dict):
    d = dict(data_dict)
    extras = _img_to_extras.get(d.get("image_t1"), {})
    if masks_from_disk:
        if "mask_t1" not in d and extras.get("mask_t1") is not None:
            d["mask_t1"] = extras["mask_t1"]
        if "mask_t2" not in d and extras.get("mask_t2") is not None:
            d["mask_t2"] = extras["mask_t2"]
    if "label" not in d:
        d["label"] = extras.get("label", 0)
    return _val_transforms_raw(d)


print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}, " f"masks_from_disk={masks_from_disk}")
if not masks_from_disk:
    print(
        "  (No disk masks found on items — using intensity-threshold CreateBrainMaskd.\n"
        "  If training ran with disk masks, the modality probe will differ from the training log.)"
    )

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, embed_dim)` pooled vectors, one per level
  - All levels are pooled from the codebook projection (embed_dim space)
  - When `channel_logits` is active, content/style split is in this embed_dim space
- `estimated_content_indices`: the Gumbel-selected embedding dim indices at level 0

For visualisation we split level-0 pooled features into **content** and **style** subsets
using `estimated_content_indices`.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []
all_content_indices = {}  # dict mapping level -> view-0 (T1) content indices
all_content_indices_v1 = {}  # dict mapping level -> view-1 (T2) content indices (only set for per-view masks)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Map modality name → view_idx for separate-encoder models.
# view 0 = T1 (first modality), view 1 = T2 (second modality).
_VIEW_IDX = {"T1": 0, "T2": 1}

# Detect per-view masks: separate encoders with channel_logits_v1
_has_per_view = (
    getattr(vqvae_module, "separate_encoders", False) and getattr(vqvae_module, "channel_logits_v1", None) is not None
)

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            # Get soft_content_masks (7th returned element)
            _, _, enc_features, _, _, _, soft_masks, *_ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
                view_idx=_VIEW_IDX.get(modality, 0),
            )

        # enc_features is a list of (1, hidden_channels) pooled tensors per level.
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())

        # Record the content indices per level.
        #  - Tuple masks: training-style n_views=2 with per-view masks (unused
        #    here since we run n_views=1, but handled for completeness).
        #  - Single-tensor mask: single-view forward.  When per-view learned
        #    masks are active the returned tensor is the view-specific mask
        #    (channel_logits_v1 when view_idx=1), so T2 must be stored in
        #    all_content_indices_v1 to preserve the per-view split.
        for lvl, mask in soft_masks.items():
            if isinstance(mask, tuple):
                mask_v0, mask_v1 = mask
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask_v0.bool())[-1].tolist()
                elif modality == "T2" and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask_v1.bool())[-1].tolist()
            else:
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask.bool())[-1].tolist()
                elif modality == "T2" and _has_per_view and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask.bool())[-1].tolist()

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print(f"\nDone. {len(all_labels)} samples total.")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")

# Derive content / style subsets from the hidden_channels features for each masked level.
all_style_indices = {}
all_style_indices_v1 = {}

for lvl in all_content_indices.keys():
    all_style_indices[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices[lvl])]
    if _has_per_view and lvl in all_content_indices_v1:
        all_style_indices_v1[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices_v1[lvl])]
    else:
        all_style_indices_v1[lvl] = all_style_indices[lvl]

if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        print(f"\n--- Level {lvl} ---")
        print(f"  content indices v0 ({len(all_content_indices[lvl])} ch): {all_content_indices[lvl][:8]}...")
        print(f"  style indices   v0 ({len(all_style_indices[lvl])} ch):   {all_style_indices[lvl][:8]}...")
        if _has_per_view and lvl in all_content_indices_v1:
            print(f"  content indices v1 ({len(all_content_indices_v1[lvl])} ch): {all_content_indices_v1[lvl][:8]}...")
            print(f"  style indices   v1 ({len(all_style_indices_v1[lvl])} ch):   {all_style_indices_v1[lvl][:8]}...")

## 7c. Content–Style Information Leakage Test

If disentanglement is working, **content should not encode modality** and
**style should not encode diagnosis**:

| Feature set | Predict modality (T1/T2) | Predict diagnosis (AD/CN/MCI) |
|---|---|---|
| Content | ~50% (chance) | **High** (anatomy encodes disease) |
| Style | **High** (contrast is modality-specific) | ~33% (chance) |

We train 5-fold cross-validated logistic regression classifiers on each feature
subset and report accuracy. Deviations from the expected pattern indicate
information leakage between content and style.

We also compute the **alignment** and **uniformity** metrics from
[Wang & Isola 2020] on the content representations, which are more principled
than visual inspection of t-SNE plots:
- **Alignment** (↓ better): expected distance between positive pairs (T1↔T2 same subject)
- **Uniformity** (↓ better): log-average of pairwise Gaussian potential (spread on hypersphere)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, Normalizer, StandardScaler

# ══════════════════════════════════════════════════════════════════
# 1. INFORMATION LEAKAGE TEST — linear (LR) + non-linear (MLP)
# Runs both probes on every per-level feature set (content / style /
# all dims) for modality AND diagnosis.
# ══════════════════════════════════════════════════════════════════

# Prepare labels
modality_labels = (all_modalities == "T2").astype(int)  # 0=T1, 1=T2
diagnosis_labels = all_labels  # integer class labels

n_classes = len(np.unique(diagnosis_labels))
chance_modality = 50.0  # binary
chance_diagnosis = 100.0 / n_classes

# Row masks for per-view indexing (T1 → v0 mask, T2 → v1 mask).
_t1_rows = all_modalities == "T1"
_t2_rows = all_modalities == "T2"


def _per_view_select(feats, idx_v0, idx_v1):
    """Apply v0 indices to T1 rows and v1 indices to T2 rows.

    Matches the training-time probe (eval/cross_reconstruction.py), which
    indexes each view's features with that view's Gumbel content mask.
    Assumes len(idx_v0) == len(idx_v1).
    """
    if list(idx_v0) == list(idx_v1):
        return feats[:, idx_v0]
    out = np.empty((feats.shape[0], len(idx_v0)), dtype=feats.dtype)
    out[_t1_rows] = feats[_t1_rows][:, idx_v0]
    out[_t2_rows] = feats[_t2_rows][:, idx_v1]
    return out


# Feature sets to test — all from embed_dim space
level0_feats = all_features["level_0"]
feature_sets = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        feature_sets[f"Content (L{lvl})"] = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
    if lc["style_idx"]:
        feature_sets[f"Style (L{lvl})"] = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
    feature_sets[f"All dims (L{lvl})"] = feats


def _lr_probe(multiclass=False):
    kw = dict(max_iter=1000, random_state=42, C=1.0)
    if multiclass:
        kw["multi_class"] = "multinomial"
    return make_pipeline(Normalizer(norm="l2"), StandardScaler(), LogisticRegression(**kw))


def _mlp_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=500,
            early_stopping=True,
            random_state=42,
        ),
    )


print("=" * 96)
print("INFORMATION LEAKAGE TEST — Linear (LR) + Non-linear (MLP), 5-fold CV")
print("=" * 96)
print(
    f"\nChance levels: modality={chance_modality:.0f}%, " f"diagnosis={chance_diagnosis:.1f}% ({n_classes} classes)\n"
)

header = (
    f"{'Feature set':<22} "
    f"{'Mod LR':>10} {'Mod MLP':>10} "
    f"{'Diag LR':>10} {'Diag MLP':>10} "
    f"{'Mod leak?':>12} {'Diag leak?':>12}"
)
print(header)
print("-" * len(header))

results = {}
for name, X in feature_sets.items():
    # Modality (binary)
    mod_lr = cross_val_score(_lr_probe(), X, modality_labels, cv=5, scoring="accuracy")
    mod_mlp = cross_val_score(_mlp_probe(), X, modality_labels, cv=5, scoring="accuracy")
    mod_lr_acc, mod_mlp_acc = mod_lr.mean() * 100, mod_mlp.mean() * 100

    # Diagnosis (multiclass)
    diag_lr = cross_val_score(_lr_probe(multiclass=True), X, diagnosis_labels, cv=5, scoring="accuracy")
    diag_mlp = cross_val_score(_mlp_probe(), X, diagnosis_labels, cv=5, scoring="accuracy")
    diag_lr_acc, diag_mlp_acc = diag_lr.mean() * 100, diag_mlp.mean() * 100

    # Leakage flags use the max over LR/MLP — any probe that picks it up is a leak.
    mod_leak = ""
    diag_leak = ""
    if "Content" in name or "All dims" in name:
        mod_leak = "YES" if max(mod_lr_acc, mod_mlp_acc) > chance_modality + 10 else "no"
    if "Style" in name:
        diag_leak = "YES" if max(diag_lr_acc, diag_mlp_acc) > chance_diagnosis + 10 else "no"

    print(
        f"{name:<22} "
        f"{mod_lr_acc:>8.1f}%  {mod_mlp_acc:>8.1f}%  "
        f"{diag_lr_acc:>8.1f}%  {diag_mlp_acc:>8.1f}%  "
        f"{mod_leak:>12} {diag_leak:>12}"
    )

    results[name] = {
        "modality_lr": mod_lr_acc,
        "modality_mlp": mod_mlp_acc,
        "diagnosis_lr": diag_lr_acc,
        "diagnosis_mlp": diag_mlp_acc,
    }

# ══════════════════════════════════════════════════════════════════
# 2. ALIGNMENT & UNIFORMITY (Wang & Isola 2020)
# ══════════════════════════════════════════════════════════════════


def compute_alignment(x, y, alpha=2):
    """Alignment: expected ||f(x) - f(y)||^alpha for positive pairs."""
    x_norm = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)
    y_norm = y / (np.linalg.norm(y, axis=1, keepdims=True) + 1e-8)
    return np.mean(np.sum((x_norm - y_norm) ** alpha, axis=1))


def compute_uniformity(x, t=2):
    """Uniformity: log of average pairwise Gaussian potential."""
    x_norm = x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)
    sq_dists = np.sum((x_norm[:, None] - x_norm[None, :]) ** 2, axis=2)
    mask = ~np.eye(len(x), dtype=bool)
    return np.log(np.mean(np.exp(-t * sq_dists[mask])))


print(f"\n{'='*70}")
print("ALIGNMENT & UNIFORMITY — Content Representations")
print("=" * 70)
print("(Lower is better for both. Alignment measures positive-pair closeness;")
print(" Uniformity measures how spread out features are on the hypersphere.)\n")

print(f"{'Feature set':<22} {'Alignment (↓)':>14} {'Uniformity (↓)':>16}")
print("-" * 55)

for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    subsets = []
    if lc["content_idx"]:
        subsets.append(
            (f"Content (L{lvl})", feats[t1_idx][:, lc["content_idx"]], feats[t2_idx][:, lc["content_idx_v1"]])
        )
    if lc["style_idx"]:
        subsets.append((f"Style (L{lvl})", feats[t1_idx][:, lc["style_idx"]], feats[t2_idx][:, lc["style_idx_v1"]]))
    subsets.append((f"All dims (L{lvl})", lc["t1"], lc["t2"]))
    for name, t1_feat, t2_feat in subsets:
        align = compute_alignment(t1_feat, t2_feat)
        combined = np.vstack([t1_feat, t2_feat])
        if len(combined) > 500:
            rng = np.random.RandomState(42)
            sub_idx = rng.choice(len(combined), 500, replace=False)
            combined = combined[sub_idx]
        unif = compute_uniformity(combined)
        print(f"{name:<22} {align:>14.4f} {unif:>16.4f}")

# ══════════════════════════════════════════════════════════════════
# 3. VISUALIZATION — LR vs MLP, grouped bars
# ══════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.35 * len(results) + 2)))

names = list(results.keys())
y_pos = np.arange(len(names))
h = 0.38
colors_base = ["tab:blue" if "Content" in n else "tab:orange" if "Style" in n else "tab:gray" for n in names]

for ax, target, chance, title in [
    (axes[0], "modality", chance_modality, "Predict MODALITY (T1 vs T2)\nContent should be at chance"),
    (axes[1], "diagnosis", chance_diagnosis, "Predict DIAGNOSIS\nStyle should be at chance"),
]:
    lr_vals = [results[n][f"{target}_lr"] for n in names]
    mlp_vals = [results[n][f"{target}_mlp"] for n in names]
    ax.barh(y_pos - h / 2, lr_vals, h, label="Linear (LR)", color=colors_base, alpha=0.9)
    ax.barh(y_pos + h / 2, mlp_vals, h, label="Non-linear (MLP)", color=colors_base, alpha=0.5, hatch="//")
    ax.axvline(chance, color="red", ls="--", label=f"Chance ({chance:.1f}%)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("Accuracy (%)")
    ax.set_title(title)
    ax.set_xlim(0, 100)
    ax.legend(loc="lower right", fontsize=8)

plt.suptitle("Content–Style Information Leakage Test — LR vs MLP", fontsize=14)
plt.tight_layout()
plt.savefig("content_style_leakage.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✓ Saved content_style_leakage.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSIS PROBE BREAKDOWN — step 1 of the content/style diagnostic
# Compare content-only, style-only, and concat(content, style) for
# predicting diagnosis, aggregated across levels. Balanced accuracy +
# macro-AUC so class imbalance doesn't confound the comparison.
# Reuses `all_features`, `level_content`, `_per_view_select`, `all_labels`
# defined in the leakage cell above.
# ══════════════════════════════════════════════════════════════════
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

diag_labels = np.asarray(all_labels)
n_classes_d = len(np.unique(diag_labels))
chance_diag = 1.0 / n_classes_d


def _probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0, random_state=42, class_weight="balanced", multi_class="multinomial"),
    )


cv_d = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Aggregate content-only and style-only features across all levels
content_parts, style_parts = [], []
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        content_parts.append(_per_view_select(feats, lc["content_idx"], lc["content_idx_v1"]))
    if lc["style_idx"]:
        style_parts.append(_per_view_select(feats, lc["style_idx"], lc["style_idx_v1"]))

X_content = np.concatenate(content_parts, axis=1) if content_parts else None
X_style = np.concatenate(style_parts, axis=1) if style_parts else None
X_both = np.concatenate([X_content, X_style], axis=1) if (X_content is not None and X_style is not None) else None

# Assemble the feature sets to probe — per-level plus aggregated
feature_sets = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        feature_sets[f"Content L{lvl}"] = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
    if lc["style_idx"]:
        feature_sets[f"Style L{lvl}"] = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
if X_content is not None:
    feature_sets["Content (all levels)"] = X_content
if X_style is not None:
    feature_sets["Style (all levels)"] = X_style
if X_both is not None:
    feature_sets["Content ⊕ Style"] = X_both

print("=" * 82)
print("DIAGNOSIS PROBE BREAKDOWN")
print(
    f"Chance balAcc = {chance_diag*100:.1f}% ({n_classes_d} classes), "
    f"N = {len(diag_labels)}, 5-fold stratified CV, balanced class weights"
)
print("=" * 82)
print(f"{'Feature set':<25} {'Dim':>5} {'BalAcc':>12} {'AUC (macro OvR)':>20}")
print("-" * 82)

probe_results = {}
for name, X in feature_sets.items():
    bal = cross_val_score(_probe(), X, diag_labels, cv=cv_d, scoring="balanced_accuracy")
    try:
        auc = cross_val_score(_probe(), X, diag_labels, cv=cv_d, scoring="roc_auc_ovr_weighted")
    except Exception:
        auc = np.array([np.nan])
    probe_results[name] = {
        "bal": bal.mean(),
        "bal_std": bal.std(),
        "auc": auc.mean(),
        "auc_std": auc.std(),
        "dim": X.shape[1],
    }
    print(
        f"{name:<25} {X.shape[1]:>5d} "
        f"{bal.mean()*100:>7.1f}% ±{bal.std()*100:>3.1f} "
        f"{auc.mean():>12.3f} ±{auc.std():>5.3f}"
    )

# ── Decision table ──────────────────────────────────────────────
print()
print("=" * 82)
print("INTERPRETATION")
print("=" * 82)
A = probe_results.get("Content (all levels)")
B = probe_results.get("Style (all levels)")
C = probe_results.get("Content ⊕ Style")
if A and B and C:
    gap_sc = B["bal"] - A["bal"]
    gain_both = C["bal"] - max(A["bal"], B["bal"])
    print(f"  Content  balAcc = {A['bal']*100:5.1f}%   (chance {chance_diag*100:.1f}%)")
    print(f"  Style    balAcc = {B['bal']*100:5.1f}%")
    print(f"  Both     balAcc = {C['bal']*100:5.1f}%")
    print(f"  Style − Content : {gap_sc*100:+.1f} pp")
    print(f"  Both   − max(A,B): {gain_both*100:+.1f} pp")
    print()
    content_near_chance = (A["bal"] - chance_diag) < 0.03
    if content_near_chance:
        print("  → Content near chance. Most likely: InfoNCE isn't pushing diagnosis")
        print("    into content. Next: decode-space probe (step 2).")
    elif gap_sc > 0.05 and gain_both > 0.03:
        print("  → Content and style carry DIFFERENT diagnostic info; combining helps.")
        print("    Real disentanglement mismatch. Next: step 2 (decode-space probe).")
    elif gap_sc > 0.05 and gain_both < 0.02:
        print("  → Style ≫ Content but combining adds nothing — same signal, wrong stream.")
        print("    Capacity/pressure issue. Next: swap-consistency loss or shrink style.")
    else:
        print("  → Content ≈ Style. Not really a disentanglement failure.")
        print("    Low priority; focus elsewhere or tighten the probe.")
else:
    print("  (need content, style, and both feature sets — check level_content indices)")

# ── Visualisation ──────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(9, 0.4 * len(probe_results) + 2))
names = list(probe_results.keys())
bals = [probe_results[n]["bal"] * 100 for n in names]
errs = [probe_results[n]["bal_std"] * 100 for n in names]
colors = [
    "tab:blue"
    if "Content" in n and "Style" not in n
    else "tab:orange"
    if "Style" in n and "Content" not in n
    else "tab:green"
    for n in names
]
ax.barh(names, bals, xerr=errs, color=colors, alpha=0.85)
ax.axvline(chance_diag * 100, color="red", ls="--", label=f"Chance ({chance_diag*100:.1f}%)")
ax.set_xlabel("Balanced accuracy (%)")
ax.set_title("Diagnosis probe — content vs style vs concat")
ax.set_xlim(0, 100)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("diagnosis_probe_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ Saved diagnosis_probe_breakdown.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSIS PROBE — follow-ups (run after the breakdown cell)
# 1. AD-vs-CN binary (drop MCI, the class that wrecks 3-way balAcc)
# 2. Nonlinear probes (MLP, RandomForest) to rule out probe capacity
# 3. Raw-volume baseline — the achievable ceiling on this N
#
# Reuses: X_content, X_style, X_both, diag_labels, _probe, cv_d, label_map,
#         items, val_transforms, all_modalities, all_subjects
#
# Note: `diag_labels` / `all_features` rows are interleaved per subject
# (subj0_T1, subj0_T2, subj1_T1, subj1_T2, ...). `X_raw` must follow the
# same ordering.
# ══════════════════════════════════════════════════════════════════
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
import torch.nn.functional as F

# ── Label distribution sanity ──────────────────────────────────
_uniq, _counts = np.unique(diag_labels, return_counts=True)
print("Class distribution (int → count):")
for _u, _c in zip(_uniq.tolist(), _counts.tolist()):
    _nm = label_names.get(_u, "?") if "label_names" in dir() else "?"
    print(f"  {_u} ({_nm}): {_c}")


# Resolve AD / CN integer labels
def _lookup(name):
    for k, v in label_map.items():
        if str(k).upper() == name:
            return v
    return None


ad_label = _lookup("AD")
cn_label = _lookup("CN")
assert ad_label is not None and cn_label is not None, f"Could not find AD/CN in {label_map}"

# Groups for CV — one group per subject so T1 and T2 of the same subject
# can't leak across folds. Fall back to row index if subjects aren't tracked.
try:
    groups_all = np.asarray(all_subjects)
    use_groups = len(groups_all) == len(diag_labels) and len(np.unique(groups_all)) > 1
except NameError:
    use_groups = False

if use_groups:
    cv_d_grp = GroupKFold(n_splits=5)
    print(f"\nUsing GroupKFold (5 splits) on subject IDs — {len(np.unique(groups_all))} unique subjects.")
else:
    cv_d_grp = cv_d
    groups_all = None
    print("\n⚠ all_subjects not available — using StratifiedKFold (within-subject T1/T2 may leak).")


def _score(clf, X, y, scoring, groups=None, cv=None):
    cv = cv if cv is not None else (cv_d_grp if use_groups else cv_d)
    return cross_val_score(clf, X, y, cv=cv, scoring=scoring, groups=groups)


# ══════════════════════════════════════════════════════════════════
# 1. AD vs CN binary probe
# ══════════════════════════════════════════════════════════════════
bin_mask = (diag_labels == ad_label) | (diag_labels == cn_label)
y_bin = (diag_labels[bin_mask] == ad_label).astype(int)
groups_bin = groups_all[bin_mask] if use_groups else None
cv_bin = GroupKFold(n_splits=5) if use_groups else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(
    f"\nBinary AD-vs-CN: N={bin_mask.sum()}  AD={int(y_bin.sum())}  "
    f"CN={int((1 - y_bin).sum())}  chance balAcc = 50.0%"
)

bin_sets = {}
if X_content is not None:
    bin_sets["Content (all)"] = X_content[bin_mask]
if X_style is not None:
    bin_sets["Style (all)"] = X_style[bin_mask]
if X_both is not None:
    bin_sets["Both"] = X_both[bin_mask]

print(f"\n{'Feature set':<16} {'LR balAcc':>12} {'LR AUC':>10}")
print("-" * 42)
for name, X in bin_sets.items():
    bal = cross_val_score(_probe(), X, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=groups_bin)
    auc = cross_val_score(_probe(), X, y_bin, cv=cv_bin, scoring="roc_auc", groups=groups_bin)
    print(f"{name:<16} {bal.mean()*100:>8.1f}% ±{bal.std()*100:.1f} " f"{auc.mean():>7.3f} ±{auc.std():.3f}")


# ══════════════════════════════════════════════════════════════════
# 2. Nonlinear probes (3-class) on content / style / both
# ══════════════════════════════════════════════════════════════════
def _mlp():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42, early_stopping=True, alpha=1e-3),
    )


def _rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)


print("\n" + "=" * 90)
print("NONLINEAR PROBE — does signal exist but not linearly?")
print("=" * 90)
print(
    f"{'Feature set':<16} {'3-cls LR':>11} {'3-cls MLP':>12} {'3-cls RF':>11} "
    f"{'AD-CN LR':>11} {'AD-CN MLP':>12} {'AD-CN RF':>11}"
)
print("-" * 90)
for name, X in [("Content (all)", X_content), ("Style (all)", X_style), ("Both", X_both)]:
    if X is None:
        continue
    g3 = groups_all if use_groups else None
    gb = groups_bin if use_groups else None

    lr3 = cross_val_score(_probe(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    mlp3 = cross_val_score(_mlp(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    rf3 = cross_val_score(_rf(), X, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()

    Xb = X[bin_mask]
    lrb = cross_val_score(_probe(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    mlpb = cross_val_score(_mlp(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    rfb = cross_val_score(_rf(), Xb, y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    print(
        f"{name:<16} {lr3*100:>8.1f}% {mlp3*100:>10.1f}% {rf3*100:>9.1f}% "
        f"{lrb*100:>8.1f}% {mlpb*100:>10.1f}% {rfb*100:>9.1f}%"
    )

# ══════════════════════════════════════════════════════════════════
# 3. Raw-volume baseline — ceiling for this dataset
# ══════════════════════════════════════════════════════════════════
# Avg-pool each modality to an 8^3 grid plus a few global stats.
# Emit rows in the SAME interleaved order as `all_features`:
#   [subj0_T1, subj0_T2, subj1_T1, subj1_T2, ...]
# so labels/groups line up 1:1.
print("\n" + "=" * 78)
print("RAW-VOLUME BASELINE (pooled voxels + global stats, no learned features)")
print("=" * 78)
print(f"Collecting raw features for {len(items)} subjects (T1 + T2 each)...")


def _vol_features(img):
    """img: (1, 1, D, H, W) float tensor on CPU."""
    with torch.no_grad():
        pooled = F.adaptive_avg_pool3d(img, (8, 8, 8)).flatten().numpy()
    brain_vox = img[img > 0]
    if brain_vox.numel() > 1:
        q_low = brain_vox.quantile(0.1)
        q_high = brain_vox.quantile(0.9)
        stats = np.array(
            [
                float((img > 0).float().sum()),
                float(brain_vox.mean()),
                float(brain_vox.std()),
                float((brain_vox < q_low).float().mean()),
                float((brain_vox > q_high).float().mean()),
            ]
        )
    else:
        stats = np.zeros(5)
    return np.concatenate([pooled, stats])


raw_feats = []
for i, it in enumerate(items):
    t = val_transforms({"image_t1": it["image"], "image_t2": it.get("z_image", it["image"])})
    for key in ("image_t1", "image_t2"):
        raw_feats.append(_vol_features(t[key].unsqueeze(0).float()))
    if (i + 1) % 50 == 0 or (i + 1) == len(items):
        print(f"  {i+1}/{len(items)}")

X_raw = np.stack(raw_feats)
print(f"Raw feature shape: {X_raw.shape}  (expected ({len(diag_labels)}, ...))")
assert X_raw.shape[0] == len(diag_labels), "X_raw rows must match diag_labels"

print(f"\n{'Probe':<6} {'3-cls balAcc':>14} {'AD-CN balAcc':>16} {'AD-CN AUC':>12}")
print("-" * 52)
for name, mk in [("LR", _probe), ("MLP", _mlp), ("RF", _rf)]:
    g3 = groups_all if use_groups else None
    gb = groups_bin if use_groups else None
    bal3 = cross_val_score(mk(), X_raw, diag_labels, cv=cv_d_grp, scoring="balanced_accuracy", groups=g3).mean()
    balb = cross_val_score(mk(), X_raw[bin_mask], y_bin, cv=cv_bin, scoring="balanced_accuracy", groups=gb).mean()
    try:
        aucb = cross_val_score(mk(), X_raw[bin_mask], y_bin, cv=cv_bin, scoring="roc_auc", groups=gb).mean()
    except Exception:
        aucb = float("nan")
    print(f"{name:<6} {bal3*100:>11.1f}%  {balb*100:>13.1f}%  {aucb:>11.3f}")

# ══════════════════════════════════════════════════════════════════
# Summary interpretation
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 78)
print("SUMMARY — how to read the numbers")
print("=" * 78)
print("  • Raw AD-vs-CN balAcc ≈ 50% → dataset/split isn't diagnosable at all.")
print("    Not a model problem. Revisit labels, N, or split.")
print("  • Raw ≫ Content, MLP ≈ LR → model is discarding signal that's in the image.")
print("    Likely capacity or objective issue; step 3 (adversary run) relevant.")
print("  • MLP ≫ LR on same features → signal is there but nonlinearly encoded.")
print("    Not a disentanglement problem; use a stronger downstream probe.")
print("  • Everything at chance including raw → your 400-subject split is underpowered")
print("    or MCI is dominating; AD-vs-CN column is the more reliable signal here.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC — reconcile notebook probe vs. training-time probe
# ══════════════════════════════════════════════════════════════════
# (A) Check whether the Gumbel content mask drifts across samples.
#     Notebook probes record the FIRST sample's mask only; if the mask
#     is stochastic in eval mode, the recorded indices are wrong for
#     the rest of the dataset and the probe result is untrustworthy.
# (B) Call eval.cross_reconstruction._collect_representations +
#     _metrics_for_level directly on the notebook's data, so the
#     only thing that can differ from the training metric is the
#     underlying sample set and transforms.
import sys
import pathlib
from types import SimpleNamespace

_ROOT = pathlib.Path().resolve()
while _ROOT != _ROOT.parent and not (_ROOT / "eval" / "cross_reconstruction.py").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from eval.cross_reconstruction import _collect_representations, _metrics_for_level

# ─────────────────────────────────────────────────────────────────
# (A) Mask drift across samples
# ─────────────────────────────────────────────────────────────────
print("=" * 70)
print("DIAGNOSTIC A — Gumbel content mask stability")
print("=" * 70)
print(f"vqvae_model.training = {vqvae_model.training}  (False = deterministic path expected)")

_n_probe = min(20, len(items))
_per_level_masks = {lvl: [] for lvl in range(nb_levels)}
for i in range(_n_probe):
    t = val_transforms({"image_t1": items[i]["image"], "image_t2": items[i]["z_image"]})
    img = t["image_t1"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, _, _, _, _, _, soft_masks, *_ = vqvae_model(img, return_recon=False, pool_only=True, view_idx=0)
    for lvl, m in soft_masks.items():
        m_t = m[0] if isinstance(m, tuple) else m
        _per_level_masks[lvl].append(tuple(torch.where(m_t.bool())[-1].tolist()))

for lvl, masks in _per_level_masks.items():
    if not masks:
        continue
    first = masks[0]
    drift = sum(1 for m in masks if m != first)
    unique = len(set(masks))
    flag = "  ⚠ STOCHASTIC (probe indices unreliable)" if drift else "  ✓ deterministic"
    print(f"  L{lvl}: {drift}/{len(masks)} samples differ from first; {unique} unique mask(s){flag}")

# ─────────────────────────────────────────────────────────────────
# (B) Training-time probe, run inline on the notebook's samples
# ─────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("DIAGNOSTIC B — training-time probe on the notebook's sample set")
print("=" * 70)

_BATCH_SIZE = 4


def _paired_batches(_items, _transforms, _batch_size):
    """Yield {'image': [t1_batch, t2_batch]} dicts, mimicking the training val loader."""
    buf_t1, buf_t2 = [], []
    for it in _items:
        t = _transforms({"image_t1": it["image"], "image_t2": it["z_image"]})
        buf_t1.append(t["image_t1"])
        buf_t2.append(t["image_t2"])
        if len(buf_t1) == _batch_size:
            yield {"image": [torch.stack(buf_t1), torch.stack(buf_t2)]}
            buf_t1, buf_t2 = [], []
    if buf_t1:
        yield {"image": [torch.stack(buf_t1), torch.stack(buf_t2)]}


_args = SimpleNamespace(
    content_style_levels=list(range(nb_levels)),
    # Fallback only — _collect_representations uses soft_content_masks when present.
    content_indices=[list(all_content_indices.get(lvl, [])) for lvl in range(nb_levels)],
)

reps = _collect_representations(
    vqvae_model,
    _paired_batches(items, val_transforms, _BATCH_SIZE),
    _args,
    DEVICE,
    max_batches=10**6,
)

print(f"Levels collected: {sorted(reps.keys())}")
for lvl, r in reps.items():
    print(
        f"  L{lvl}: content_v0 {r['content_v0'].shape}, content_v1 {r['content_v1'].shape}, "
        f"style_v0 {r['style_v0'].shape}, style_v1 {r['style_v1'].shape}"
    )

print()
print("Training-probe metrics (apples-to-apples with eval/cross_reconstruction.py):")
print(f"{'Level':<8} {'modality_probe_acc':>22} {'modality_invariance':>22}")
print("-" * 54)
for lvl in sorted(reps.keys()):
    m = _metrics_for_level(reps[lvl])
    print(f"  L{lvl:<5} {m['content/modality_probe_acc']:>22.3f} " f"{m['content/modality_invariance']:>22.3f}")

print()
print("If DIAGNOSTIC B matches the training log (~0.6) but the cell above still")
print("reports ~0.8, the gap is purely how the notebook assembles features")
print("(single-view forward, first-sample-only mask, stochastic Gumbel, etc).")
print("If DIAGNOSTIC B ALSO reports ~0.8, the gap is the sample set / checkpoint,")
print("not a code bug — compare `items` here to the val split used during training.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DIAGNOSTIC C — sweep sample size N vs. modality probe accuracy
# ══════════════════════════════════════════════════════════════════
# Logistic-regression probe accuracy grows with N. If the training log's
# ~0.6 was simply under-resolved (small val N) and the true leakage is
# ~0.8, this sweep will show the curve rising from ~0.6 at small N to
# ~0.8 as N grows. If accuracy is roughly flat across N, the gap is NOT
# sample size (→ suspect split/checkpoint/transforms instead).

import numpy as np
import matplotlib.pyplot as plt

_N_total = len(items)
_N_grid = sorted({n for n in [25, 50, 100, 200, 400, _N_total] if n <= _N_total and n >= 10})
print(f"Sweeping N ∈ {_N_grid}  (total available: {_N_total})")

_rng = np.random.RandomState(0)
_perm = _rng.permutation(_N_total)  # fixed shuffle so subsets are nested-ish, not modality-ordered

_sweep = {lvl: [] for lvl in range(nb_levels)}

for _N in _N_grid:
    _sub_items = [items[i] for i in _perm[:_N]]
    _reps_N = _collect_representations(
        vqvae_model,
        _paired_batches(_sub_items, val_transforms, _BATCH_SIZE),
        _args,
        DEVICE,
        max_batches=10**6,
    )
    for lvl in _reps_N:
        m = _metrics_for_level(_reps_N[lvl])
        _sweep[lvl].append((m["content/modality_probe_acc"], m["content/modality_invariance"]))
    # Report progress
    line = f"  N={_N:>4}"
    for lvl in sorted(_reps_N.keys()):
        line += f"   L{lvl}: {_sweep[lvl][-1][0]:.3f}"
    print(line)

# ── Plot ──
fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
for lvl, pts in _sweep.items():
    if not pts:
        continue
    accs = [p[0] for p in pts]
    ax.plot(_N_grid, accs, marker="o", label=f"Level {lvl}")
ax.axhline(0.5, color="gray", ls=":", label="chance")
ax.set_xscale("log")
ax.set_xlabel("N (paired subjects)")
ax.set_ylabel("Content → modality probe accuracy")
ax.set_title("Modality-leakage probe vs. sample size\n(matches training probe code; varies N)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("modality_probe_vs_N.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved modality_probe_vs_N.png")

print()
print("Interpretation:")
print("  • Monotone rise → the training log's low number was under-resolved;")
print("    the true leakage at this checkpoint is the high-N value.")
print("  • Flat curve    → gap is NOT sample size. Check split / checkpoint /")
print("    transforms (diagnostic B already rules out code-path differences).")

## 9. Diagnostic label prediction

For each feature set (all encoder levels + level-0 content/style subsets) we train a
**logistic regression** and a **random forest** classifier on T1 features only (so modality
doesn't leak into the score), using stratified 5-fold cross-validation.

This tells us:
- **Which hierarchical level** encodes the most diagnostically-relevant information
- **Whether content or style channels** at level 0 carry the diagnostic signal
- How close any level is to chance (baseline = majority-class rate)

Results are shown as a bar chart and a summary table.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier

# ── use T1 features only so modality can't trivially distinguish classes ──────
t1_mask = all_modalities == "T1"
y = all_labels[t1_mask]

# Build the feature sets to evaluate
feature_sets = {}
for i in range(nb_levels):
    feature_sets[f"Level {i} (all dims)"] = all_features[f"level_{i}"][t1_mask]

# Level-0 content / style subsets
if 0 in all_content_indices and len(all_style_indices.get(0, [])) > 0:
    level0_feats = all_features["level_0"]
    feature_sets["Level 0 — content"] = level0_feats[t1_mask][:, all_content_indices[0]]
    feature_sets["Level 0 — style"] = level0_feats[t1_mask][:, all_style_indices[0]]


# ── classifiers ───────────────────────────────────────────────────────────────
def make_lr():
    """Logistic Regression with standard scaling."""
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
        ]
    )


def make_rf():
    """Random Forest (no scaling needed)."""
    return RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=cv).mean()

print(f"{'Feature set':<28} {'LR acc':>8} {'LR ±':>7}  {'RF acc':>8} {'RF ±':>7}")
print("-" * 65)

results = []
for name, X in feature_sets.items():
    lr_scores = cross_val_score(make_lr(), X, y, cv=cv, scoring="accuracy")
    rf_scores = cross_val_score(make_rf(), X, y, cv=cv, scoring="accuracy")
    results.append(
        {
            "Feature set": name,
            "LR mean": lr_scores.mean(),
            "LR std": lr_scores.std(),
            "RF mean": rf_scores.mean(),
            "RF std": rf_scores.std(),
        }
    )
    print(
        f"{name:<28} {lr_scores.mean():.3f}   ±{lr_scores.std():.3f}   "
        f"{rf_scores.mean():.3f}   ±{rf_scores.std():.3f}"
    )

print("-" * 65)
print(f"{'Majority-class baseline':<28} {chance:.3f}")

results_df = pd.DataFrame(results)
print("\nSummary dataframe saved as  results_df  (use .to_csv() to export)")

In [ ]:
# ── Bar chart: LR and RF accuracy per feature set ────────────────────────────
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(results_df) * 1.6), 5))

bars_lr = ax.bar(
    x - width / 2,
    results_df["LR mean"],
    width,
    yerr=results_df["LR std"],
    capsize=4,
    label="Logistic Regression",
    color="steelblue",
    alpha=0.85,
)

bars_rf = ax.bar(
    x + width / 2,
    results_df["RF mean"],
    width,
    yerr=results_df["RF std"],
    capsize=4,
    label="Random Forest",
    color="tomato",
    alpha=0.85,
)

# Chance-level line
ax.axhline(chance, color="grey", linestyle="--", linewidth=1.2, label=f"Chance ({chance:.2f})")

# Annotate bars with their mean accuracy
for bar in list(bars_lr) + list(bars_rf):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005, f"{h:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(results_df["Feature set"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("5-fold CV accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Diagnostic label prediction accuracy per VQ-VAE-2 feature set\n(T1 features only, 5-fold stratified CV)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("diagnostic_prediction_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved diagnostic_prediction_accuracy.png")

# ── Per-class breakdown (confusion matrix for best feature set) ──────────────
best_name = results_df.loc[results_df["RF mean"].idxmax(), "Feature set"]
best_X = feature_sets[best_name]
print(f"\nBest feature set by RF accuracy: '{best_name}'")
print("Showing confusion matrix (RF, 5-fold average) ...")

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_pred = cross_val_predict(make_rf(), best_X, y, cv=cv)
cm = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[label_names[i] for i in sorted(label_names)])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion matrix — RF on '{best_name}'")
plt.tight_layout()
plt.savefig("confusion_matrix_best_level.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrix_best_level.png")

print("\nClassification report:")
print(classification_report(y, y_pred, target_names=[label_names[i] for i in sorted(label_names)]))

## 17. Demographic Probes — Sex & Race on Content vs Style

Train **linear** (logistic regression) and **non-linear** (MLP) probes to predict
**sex** and **race** from content-only, style-only, and concatenated features at each
level plus aggregates. Uses T1 scans only so modality does not confound the comparison.

If content captures shared anatomy, demographic traits visible in anatomy (sex) should
be predictable from content. If style captures modality-specific contrast, demographics
should ideally not be encoded there — any strong signal is leakage.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# DEMOGRAPHIC PROBES — sex & race on content vs style features
# Linear (logistic regression) + non-linear (MLP) classifiers, T1 only
# to avoid modality confound. Reuses all_features, level_content,
# all_subjects, all_modalities, _per_view_select, nb_levels.
# ══════════════════════════════════════════════════════════════════
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.dummy import DummyClassifier

# ── Load demographics and align with all_subjects / all_modalities ───────────
_demo_df = pd.read_csv(os.path.join(os.path.dirname(CSV_PATH), "merged_data.csv"))
_demo_lookup = _demo_df.drop_duplicates(subset="Subject").set_index("Subject")

_gender_map = {1.0: "Male", 2.0: "Female"}
_race_map = {
    1: "Am Indian/Alaskan",
    2: "Asian",
    3: "Native Hawaiian/Pacific Isl",
    4: "Black",
    5: "White",
    6: "More than one",
    7: "Unknown",
}


def _lookup_gender(subj):
    if subj in _demo_lookup.index:
        v = _demo_lookup.loc[subj, "PTGENDER"]
        return _gender_map.get(float(v)) if pd.notna(v) else None
    return None


def _lookup_race(subj):
    if subj in _demo_lookup.index:
        v = _demo_lookup.loc[subj, "PTRACCAT"]
        if pd.isna(v):
            return None
        first = str(v).split("|")[0]
        try:
            return _race_map.get(int(first))
        except ValueError:
            return None
    return None


_sex_all = np.array([_lookup_gender(s) for s in all_subjects], dtype=object)
_race_all = np.array([_lookup_race(s) for s in all_subjects], dtype=object)

# T1 only + drop missing demographics
_t1_mask = all_modalities == "T1"

# ── Feature sets (per-level content/style + aggregates) ──────────────────────
_content_parts, _style_parts = [], []
_feature_sets_all = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    if lc["content_idx"]:
        X_c = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"])
        _feature_sets_all[f"Content L{lvl}"] = X_c
        _content_parts.append(X_c)
    if lc["style_idx"]:
        X_s = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"])
        _feature_sets_all[f"Style L{lvl}"] = X_s
        _style_parts.append(X_s)
    _feature_sets_all[f"All L{lvl}"] = feats
if _content_parts:
    _feature_sets_all["Content (all levels)"] = np.concatenate(_content_parts, axis=1)
if _style_parts:
    _feature_sets_all["Style (all levels)"] = np.concatenate(_style_parts, axis=1)
if _content_parts and _style_parts:
    _feature_sets_all["Content ⊕ Style"] = np.concatenate(
        [_feature_sets_all["Content (all levels)"], _feature_sets_all["Style (all levels)"]], axis=1
    )


# ── Probe factories ──────────────────────────────────────────────────────────
def _lr_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=1.0, random_state=42, class_weight="balanced"),
    )


def _mlp_probe():
    return make_pipeline(
        Normalizer(norm="l2"),
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=500,
            early_stopping=True,
            random_state=42,
        ),
    )


_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Run probes for each target ───────────────────────────────────────────────
_targets = {"Sex": _sex_all, "Race": _race_all}
demographic_probe_results = {}

for target_name, y_all in _targets.items():
    valid = _t1_mask & np.array([v is not None for v in y_all])
    y = y_all[valid].astype(str)
    classes, counts = np.unique(y, return_counts=True)
    # Drop rare classes (< 2*n_splits) that would break stratified CV
    keep_classes = classes[counts >= 2 * _cv.n_splits]
    mask_keep = np.isin(y, keep_classes)
    y_str = y[mask_keep]
    valid_idx = np.where(valid)[0][mask_keep]

    # MLPClassifier's early-stopping scorer calls np.isnan on predictions,
    # which fails on object/string labels. Encode to ints.
    _le = LabelEncoder()
    y = _le.fit_transform(y_str)
    class_names = list(_le.classes_)

    n_classes = len(np.unique(y))
    chance = cross_val_score(DummyClassifier(strategy="most_frequent"), np.zeros((len(y), 1)), y, cv=_cv).mean()

    print("=" * 86)
    print(f"DEMOGRAPHIC PROBE — Target: {target_name}")
    print(f"N = {len(y)} (T1 only, after dropping missing / rare classes)")
    _labels, _cnts = np.unique(y_str, return_counts=True)
    print(f"Classes ({n_classes}): {dict(zip(_labels, _cnts))}")
    print(f"Majority-class baseline = {chance*100:.1f}%")
    print("=" * 86)
    print(f"{'Feature set':<25} {'Dim':>5} {'LR balAcc':>14} {'MLP balAcc':>14}")
    print("-" * 86)

    per_target = {}
    for name, X_full in _feature_sets_all.items():
        X = X_full[valid_idx]
        lr_sc = cross_val_score(_lr_probe(), X, y, cv=_cv, scoring="balanced_accuracy")
        mlp_sc = cross_val_score(_mlp_probe(), X, y, cv=_cv, scoring="balanced_accuracy")
        per_target[name] = {
            "lr": lr_sc.mean(),
            "lr_std": lr_sc.std(),
            "mlp": mlp_sc.mean(),
            "mlp_std": mlp_sc.std(),
            "dim": X.shape[1],
        }
        print(
            f"{name:<25} {X.shape[1]:>5d} "
            f"{lr_sc.mean()*100:>8.1f}% ±{lr_sc.std()*100:>3.1f} "
            f"{mlp_sc.mean()*100:>8.1f}% ±{mlp_sc.std()*100:>3.1f}"
        )
    per_target["_chance"] = chance
    demographic_probe_results[target_name] = per_target
    print()

# ── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(_targets), figsize=(8 * len(_targets), 0.35 * len(_feature_sets_all) + 2))
if len(_targets) == 1:
    axes = [axes]

for ax, (target_name, per_target) in zip(axes, demographic_probe_results.items()):
    chance = per_target.pop("_chance")
    names = list(per_target.keys())
    lr_vals = [per_target[n]["lr"] * 100 for n in names]
    mlp_vals = [per_target[n]["mlp"] * 100 for n in names]
    lr_err = [per_target[n]["lr_std"] * 100 for n in names]
    mlp_err = [per_target[n]["mlp_std"] * 100 for n in names]
    y_pos = np.arange(len(names))
    h = 0.38
    ax.barh(y_pos - h / 2, lr_vals, h, xerr=lr_err, label="Linear (LR)", color="tab:blue", alpha=0.85)
    ax.barh(y_pos + h / 2, mlp_vals, h, xerr=mlp_err, label="Non-linear (MLP)", color="tab:orange", alpha=0.85)
    ax.axvline(chance * 100, color="red", ls="--", label=f"Majority ({chance*100:.1f}%)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_xlabel("Balanced accuracy (%)")
    ax.set_title(f"Predict {target_name}")
    ax.set_xlim(0, 100)
    ax.legend(loc="lower right", fontsize=8)
    # restore chance for downstream inspection
    per_target["_chance"] = chance

plt.suptitle("Demographic Probes — Content vs Style (T1 only, 5-fold CV)", fontsize=13)
plt.tight_layout()
plt.savefig("demographic_probes_sex_race.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Saved demographic_probes_sex_race.png")
print("  Results available in `demographic_probe_results` dict.")

## 18. L0 Leakage Diagnostics

Three checks to determine whether low `sep_L0` is **real modality leakage** at L0 or just **metric asymmetry** (anatomy info naturally lives at coarser levels).

1. **Modality probe per level** — directly measures how much T1-vs-T2 info each content space carries. Pure leakage signal.
2. **Normalized separation** — `(anatomy − modality) / (full_anatomy − chance)` per level, to quotient out each level's anatomy headroom.
3. **L0-ablation proxy** — compare anatomy accuracy using `Content L0 only` vs `Content L1+L2 only` vs `Content all`. If L0-only is near chance while L1+L2 ≈ full, L0 content isn't carrying anatomy.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# L0 LEAKAGE DIAGNOSTICS
# Requires cell 69 to have run (uses _feature_sets_all, _targets,
# demographic_probe_results, _t1_mask, _cv, _lr_probe, level_content,
# all_features, all_modalities, nb_levels, _per_view_select, _sex_all,
# _race_all).
# ══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier

_cv_all = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def _probe_balAcc(X, y):
    """Mean balanced accuracy across 5-fold CV for linear probe on X, y."""
    return cross_val_score(_lr_probe(), X, y, cv=_cv_all, scoring="balanced_accuracy").mean()


# ── Diagnostic 1: modality probe per level (T1+T2 rows, labelled by view) ───
modality_y = (all_modalities == "T2").astype(int)
modality_chance = cross_val_score(
    DummyClassifier(strategy="most_frequent"),
    np.zeros((len(modality_y), 1)),
    modality_y,
    cv=_cv_all,
    scoring="balanced_accuracy",
).mean()

modality_probe_per_level = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    feats = all_features[f"level_{lvl}"]
    content_X = _per_view_select(feats, lc["content_idx"], lc["content_idx_v1"]) if lc["content_idx"] else None
    style_X = _per_view_select(feats, lc["style_idx"], lc["style_idx_v1"]) if lc["style_idx"] else None

    row = {"chance": modality_chance}
    if content_X is not None:
        row["content"] = _probe_balAcc(content_X, modality_y)
    if style_X is not None:
        row["style"] = _probe_balAcc(style_X, modality_y)
    row["all"] = _probe_balAcc(feats, modality_y)
    modality_probe_per_level[lvl] = row

print("=" * 78)
print("DIAGNOSTIC 1 — Modality probe per level (T1 vs T2, balanced accuracy)")
print(f"Chance (majority-class) baseline = {modality_chance*100:.1f}%")
print("=" * 78)
print(f"{'Level':<8}{'content':>12}{'style':>12}{'all':>12}{'leakage':>14}")
print("-" * 78)
for lvl, row in modality_probe_per_level.items():
    leakage = row.get("content", np.nan) - modality_chance
    print(
        f"L{lvl:<7}"
        f"{row.get('content', float('nan'))*100:>11.1f}%"
        f"{row.get('style', float('nan'))*100:>11.1f}%"
        f"{row.get('all', float('nan'))*100:>11.1f}%"
        f"{leakage*100:>+13.1f}pp"
    )
print(
    "  → content column above chance = modality info leaking into content. "
    "Compare L0 vs L1/L2: if L0 is noticeably higher, leakage is real."
)


# ── Diagnostic 2: normalized separation per level ────────────────────────────
# sep_norm_L = (mean_anatomy_acc - modality_acc) / (full_anatomy_acc - chance)
# Uses existing demographic_probe_results from cell 69 for anatomy (Sex, Race).
# Falls back to recomputing if a target is missing.
print()
print("=" * 78)
print("DIAGNOSTIC 2 — Normalized separation per level (anatomy vs modality)")
print("=" * 78)
print(f"{'Level':<8}{'anat(content)':>16}{'anat(full)':>14}{'mod(content)':>16}{'sep':>10}{'sep_norm':>12}")
print("-" * 78)

sep_table = {}
for lvl in range(nb_levels):
    anat_content = []
    anat_full = []
    for target_name in _targets:  # Sex, Race
        per = demographic_probe_results.get(target_name, {})
        content_key = f"Content L{lvl}"
        all_key = f"All L{lvl}"
        chance_key = per.get("_chance", np.nan)
        if content_key in per:
            anat_content.append(per[content_key]["lr"])
        if all_key in per:
            anat_full.append(per[all_key]["lr"])
    if not anat_content or not anat_full:
        continue
    anat_content_mean = float(np.mean(anat_content))
    anat_full_mean = float(np.mean(anat_full))
    anat_chance_mean = float(np.mean([demographic_probe_results[t]["_chance"] for t in _targets]))
    mod_content = modality_probe_per_level[lvl].get("content", np.nan)

    sep = anat_content_mean - mod_content
    headroom = anat_full_mean - anat_chance_mean
    sep_norm = sep / headroom if headroom > 1e-6 else np.nan
    sep_table[lvl] = {
        "anat_content": anat_content_mean,
        "anat_full": anat_full_mean,
        "mod_content": mod_content,
        "sep": sep,
        "sep_norm": sep_norm,
    }
    print(
        f"L{lvl:<7}"
        f"{anat_content_mean*100:>15.1f}%"
        f"{anat_full_mean*100:>13.1f}%"
        f"{mod_content*100:>15.1f}%"
        f"{sep*100:>+9.1f}"
        f"{sep_norm:>12.2f}"
    )
print(
    "  → if L0 sep_norm is close to L1/L2 sep_norm, the raw L0 sep gap is "
    "just anatomy headroom — not real leakage. If L0 sep_norm is much lower, "
    "L0 is genuinely leaking."
)


# ── Diagnostic 3: L0-ablation proxy ──────────────────────────────────────────
# Compare anatomy probe accuracy for (a) L0-only content, (b) L1+L2-only
# content, (c) all-levels content. If L0-only ≈ chance and L1+L2 ≈ all, then
# L0 content isn't carrying anatomy signal — consistent with "L0 content is
# mostly modality".
print()
print("=" * 78)
print("DIAGNOSTIC 3 — L0-ablation proxy (anatomy balanced accuracy per scope)")
print("=" * 78)

# Build the three feature sets on T1-only rows.
_content_by_level = {}
for lvl in range(nb_levels):
    lc = level_content[lvl]
    if lc["content_idx"]:
        _content_by_level[lvl] = _per_view_select(all_features[f"level_{lvl}"], lc["content_idx"], lc["content_idx_v1"])

_scopes = {}
if 0 in _content_by_level:
    _scopes["L0 only"] = _content_by_level[0]
_higher = [_content_by_level[l] for l in range(1, nb_levels) if l in _content_by_level]
if _higher:
    _scopes["L1+L2 only"] = np.concatenate(_higher, axis=1)
if _content_by_level:
    _scopes["all levels"] = np.concatenate([_content_by_level[l] for l in sorted(_content_by_level)], axis=1)

ablation_results = {}
for target_name, y_all in _targets.items():
    valid = _t1_mask & np.array([v is not None for v in y_all])
    y = y_all[valid].astype(str)
    classes, counts = np.unique(y, return_counts=True)
    keep = classes[counts >= 2 * _cv_all.n_splits]
    mask_keep = np.isin(y, keep)
    y_str = y[mask_keep]
    valid_idx = np.where(valid)[0][mask_keep]
    y_enc = LabelEncoder().fit_transform(y_str)

    chance = cross_val_score(
        DummyClassifier(strategy="most_frequent"),
        np.zeros((len(y_enc), 1)),
        y_enc,
        cv=_cv_all,
        scoring="balanced_accuracy",
    ).mean()

    print(f"\nTarget: {target_name}  (chance = {chance*100:.1f}%)")
    print(f"  {'Scope':<14}{'balAcc':>10}{'Δ vs chance':>16}")
    print("  " + "-" * 40)
    row = {"_chance": chance}
    for scope_name, X_full in _scopes.items():
        X = X_full[valid_idx]
        acc = _probe_balAcc(X, y_enc)
        row[scope_name] = acc
        print(f"  {scope_name:<14}{acc*100:>9.1f}%{(acc-chance)*100:>+15.1f}pp")
    ablation_results[target_name] = row

print(
    "\n  → L0-only near chance + L1+L2 ≈ all-levels  ⇒  L0 content is mostly "
    "noise/modality; upper levels carry anatomy. Consider shrinking C_content_L0."
)


# ── Visualisation ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))

# (a) Modality probe per level — leakage bar
ax = axes[0]
lvls = sorted(modality_probe_per_level)
content_vals = [modality_probe_per_level[l].get("content", np.nan) * 100 for l in lvls]
style_vals = [modality_probe_per_level[l].get("style", np.nan) * 100 for l in lvls]
x = np.arange(len(lvls))
w = 0.35
ax.bar(x - w / 2, content_vals, w, label="content", color="tab:blue")
ax.bar(x + w / 2, style_vals, w, label="style", color="tab:orange")
ax.axhline(modality_chance * 100, color="red", ls="--", label=f"chance ({modality_chance*100:.1f}%)")
ax.set_xticks(x)
ax.set_xticklabels([f"L{l}" for l in lvls])
ax.set_ylabel("Modality balAcc (%)")
ax.set_title("Diagnostic 1 — modality probe per level")
ax.legend()

# (b) sep vs sep_norm per level
ax = axes[1]
lvls2 = sorted(sep_table)
sep_vals = [sep_table[l]["sep"] * 100 for l in lvls2]
sep_norm_vals = [sep_table[l]["sep_norm"] * 100 for l in lvls2]
x = np.arange(len(lvls2))
ax.bar(x - w / 2, sep_vals, w, label="raw sep", color="tab:blue")
ax.bar(x + w / 2, sep_norm_vals, w, label="sep_norm ×100", color="tab:green")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"L{l}" for l in lvls2])
ax.set_ylabel("%")
ax.set_title("Diagnostic 2 — raw vs normalized separation")
ax.legend()

# (c) L0 ablation proxy
ax = axes[2]
targets_plot = list(ablation_results.keys())
scopes = [k for k in _scopes]
x = np.arange(len(targets_plot))
w = 0.8 / max(len(scopes), 1)
for i, scope in enumerate(scopes):
    vals = [ablation_results[t].get(scope, np.nan) * 100 for t in targets_plot]
    ax.bar(x + (i - (len(scopes) - 1) / 2) * w, vals, w, label=scope)
for i, t in enumerate(targets_plot):
    ax.hlines(ablation_results[t]["_chance"] * 100, i - 0.4, i + 0.4, colors="red", linestyles="dashed")
ax.set_xticks(x)
ax.set_xticklabels(targets_plot)
ax.set_ylabel("Anatomy balAcc (%)")
ax.set_title("Diagnostic 3 — L0-ablation proxy\n(red dashes = chance)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("l0_leakage_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✓ Saved l0_leakage_diagnostics.png")
print("  Results: `modality_probe_per_level`, `sep_table`, `ablation_results`")